# Exercise 1: Hydrogenation of bimetallic nanoparticles

## Overview

The hydrogen storage capacity of metallic nanoparticles depends on three key
structural parameters: **size**, **shape**, and **chemical composition**.
For AuPd bimetallic systems, even small changes in the Au:Pd ratio or in the
degree of chemical ordering can drastically change how much hydrogen the particle
absorbs at a given temperature and pressure.

In this exercise we use **grand-canonical Monte Carlo (GCMC)** to simulate
hydrogen adsorption on AuPd nanoparticles. The chemical potential $\mu$ of the
hydrogen reservoir controls the H loading, and by sweeping $\mu$ we can build
the adsorption isotherm $\langle N_H \rangle(\mu)$.

The interatomic interactions are described by a **GAP machine-learning
potential** trained on DFT data for the Au–Pd–H system, which gives
near-DFT accuracy at a fraction of the cost.

## Goals

1. Generate an AuPd nanoparticle using a realistic "cooking" recipe:
   anneal at high temperature → quench → relax → anneal.
2. Run GCMC to hydrogenate the cooked nanoparticle and monitor H uptake.
3. Compare your cooked structure against two reference structures:
   an ordered cuboctahedron and a disordered one.
4. Analyse energy and H-concentration convergence.
5. Characterise H adsorption sites by coordination environment.
6. Explore the effect of GCMC parameters on the result.

## `mc.log` column reference

The `mc.log` file written by TurboGAP has the following columns:

| Column | Quantity |
|--------|----------|
| 0 | MC step |
| 1 | Move type |
| 2 | Accepted (1/0) |
| 3 | Trial energy (eV) |
| 4 | Current energy (eV) |
| 5 | Total number of atoms |
| 6 | Species label |
| 7 | Number of H atoms |


---
## Background — Nanoparticle stability and structure–property relationships

### 1. Why nanoparticles behave differently from bulk

A bulk metal crystal has a well-defined repeating structure where the vast
majority of atoms sit in the interior. As the particle shrinks, an increasing
fraction of atoms sits at the **surface**, at **edges**, or at **corners** —
sites that are geometrically and electronically very different from interior
bulk sites.

The fraction of surface atoms $f_s$ scales roughly as:

$$f_s \approx \frac{N_s}{N_{\mathrm{tot}}} \propto N_{\mathrm{tot}}^{-1/3}
\propto \frac{1}{d}$$

where $d$ is the particle diameter. For a 1 nm particle, nearly all atoms are
at the surface; for a 10 nm particle, only ~10% are. This has direct
consequences for:

- **Thermodynamic stability** — surface atoms have fewer neighbours and higher
  energy, so surface energy $\gamma$ dominates the total free energy at small
  sizes.
- **Melting point depression** — the Gibbs–Thomson relation predicts that the
  melting temperature decreases as $\sim 1/d$. For a 1–2 nm AuPd particle,
  the effective melting temperature can be several hundred kelvin below bulk,
  which is why annealing at 800 K is sufficient to induce significant atomic
  mobility without vaporising the cluster.
- **Catalytic and adsorption properties** — active sites are predominantly
  at the surface, so surface area per unit mass increases as $\sim 1/d$.

### 2. Magic sizes and geometric shells

Nanoparticles are most stable at specific "**magic**" atom counts corresponding
to complete geometric shells. For FCC metals:

| Shape | Magic sizes $N$ |
|-------|----------------|
| Cuboctahedron (CO) | 13, **55**, 147, 309, 561 |
| Truncated octahedron (TO) | 38, **79**, 201, 405 |
| Icosahedron | 13, 55, 147, 309 |

In this exercise we work with $N = 55$ atoms, a magic size for both the
cuboctahedron and the icosahedron.

The **cuboctahedron** exposes (111) and (100) facets with equal area.
It has 12 nearest neighbours in the bulk-like interior and well-defined
terrace, edge, and corner sites on the surface.
The cohesive energy per atom scales as:

$$\frac{E_c}{N}(\mathrm{NP}) \approx \frac{E_c}{N}(\mathrm{bulk})
\left(1 - \frac{c}{N^{1/3}}\right)$$

Larger particles are thermodynamically more stable — but also less reactive
because the proportion of low-coordination surface sites decreases.

A useful characterisation of surface sites is their **coordination number**
(CN), i.e. the number of nearest metal neighbours:

| Site | CN (FCC surface) | Fraction in $N=55$ CO |
|------|-----------------|----------------------|
| Corner | 5 | 12 atoms |
| Edge | 7 | 24 atoms |
| (100) terrace | 8 | 6 atoms |
| (111) terrace | 9 | 8 atoms |
| Subsurface / bulk | 12 | 5 atoms |

### 3. Chemical ordering in bimetallic nanoparticles

For a bimetallic AuPd particle, beyond size and shape there is a third
degree of freedom: **how the two elements are distributed**:

| Ordering | Description | Driving force |
|----------|-------------|--------------|
| **Random alloy** | Random Au/Pd occupancy | Configurational entropy (high $T$) |
| **Core–shell** | Au-rich shell, Pd-rich core | Surface energy: $\gamma_{\mathrm{Au}} < \gamma_{\mathrm{Pd}}$ |
| **Ordered intermetallic** | Au/Pd alternate on sublattice | Mixing enthalpy (low $T$) |

In Au–Pd systems, **Au segregates to the surface** because it has lower
surface energy and a larger atomic radius. At high temperatures configurational
entropy favours mixing. The melt–quench–anneal protocol explores this
competition: high-$T$ annealing allows the system to sample many orderings
while slow cooling (anneal) lets it relax towards a low-energy state.

### 4. Hydrogen adsorption sites and coordination

Hydrogen does not adsorb equally everywhere on a nanoparticle surface.
The binding energy depends on the **local coordination environment**:

| Site type | Metal neighbours | Preferred facet |
|-----------|----------------|----------------|
| **On-top** | 1 | any |
| **Bridge** | 2 | any |
| **3-fold hollow (fcc)** | 3 | (111) |
| **3-fold hollow (hcp)** | 3 | (111) |
| **4-fold hollow** | 4 | (100) |
| **Subsurface** | ≥ 5 (bulk-like) | — |

Generally, binding is **stronger at low-coordination sites** (edges, corners)
because the metal d-band is narrower and shifted up in energy, increasing
the bonding interaction with H 1s. On Pd, hollow sites on (111) facets
are typically most stable ($E_b \approx -0.4$ to $-0.6$ eV), while
Au binds H much more weakly ($E_b \approx +0.1$ to $+0.3$ eV on terraces).

For AuPd alloys two additional effects operate:

- **Ligand effect**: a Pd atom surrounded by Au neighbours has a modified
  d-band centre relative to pure Pd — typically a downward shift that
  *weakens* H binding.
- **Ensemble effect**: H$_2$ dissociation requires at least two adjacent Pd
  atoms. An isolated Pd atom (fully surrounded by Au) cannot dissociate H$_2$,
  while a Pd ensemble of three or more atoms can.

These two effects compete: the ensemble effect tends to favour disordered
particles (more Pd–Pd contact), while the ligand effect can either strengthen
or weaken binding depending on the local environment.

### 5. Connecting structure to GCMC output

In the GCMC simulation:

- The **chemical potential** $\mu$ maps to the H$_2$ partial pressure:
  $\mu_{\mathrm{H}} = \frac{1}{2}(\mu_{\mathrm{H_2}}^0 + k_BT\ln p_{\mathrm{H_2}})$.
  More negative $\mu$ corresponds to lower pressure (fewer H atoms).
- Sites with stronger binding fill **first** (at the most negative $\mu$);
  weaker sites fill as $\mu$ increases.
- The **equilibrium H concentration** $\langle N_H\rangle / N_{\mathrm{metal}}$
  at fixed $\mu$ reflects the thermally averaged occupation of all
  accessible binding sites.
- Differences between ordered and disordered particles directly reflect
  differences in the **distribution of site binding energies** caused by
  changes in local chemical environment.

> **Key question for Part 2:** when you compare ordered and disordered
> particles, ask yourself — does the disordered particle adsorb *more* or
> *fewer* H atoms at the same $\mu$? Is this consistent with the ligand
> effect, the ensemble effect, or both?


---
## Setup

Unpack the GAP potential files and import the required Python packages.
Run the cell below once at the start of the session.


In [ ]:
import subprocess
subprocess.run(["tar", "-xzf", "gap_files.tar.xz"])

from ase.io import write, read
from weas_widget import WeasWidget
import numpy as np
import matplotlib.pyplot as plt


---
## Part 1 — Nanoparticle preparation ("cooking")

A realistic nanoparticle is not a perfect crystal fragment.
We mimic experimental synthesis through a four-step thermal protocol that
allows the particle to explore many chemical orderings and settle into a
low-energy disordered state:

| Step | Script | $T$ | Purpose |
|------|--------|-----|---------|
| 1. Initial structure | `make_NP.py` | — | Build a perfect cuboctahedron |
| 2. Anneal | `1.anneal.sh` | 800 K | Activate atomic mobility, randomise ordering |
| 3. Quench | `2.quench.sh` | 800→100 K | Freeze in a disordered state |
| 4. Relax | `3.relax.sh` | — | Remove residual forces |
| 5. Anneal | `4.anneal2.sh` | 600 K | Allow local relaxation |

Each step reads the output of the previous one.
Track how the energy per atom evolves — it should decrease at each step.


### Step 1 — Initial structure

Run `python make_NP.py` in the terminal to generate the starting cuboctahedron.
The script places Au and Pd atoms on an FCC lattice in a 1:1 ratio ($N=55$).


In [ ]:
atoms = read("structures/atoms0.xyz")
N_atoms = len(atoms)
print(f"Composition : {atoms.get_chemical_formula()}")
print(f"Total atoms : {N_atoms}")
print(f"Au atoms    : {atoms.get_chemical_symbols().count('Au')}")
print(f"Pd atoms    : {atoms.get_chemical_symbols().count('Pd')}")

viewer = WeasWidget()
viewer.avr.model_style = 0
viewer.from_ase(atoms)
viewer


### Step 2 — Anneal at high temperature (800 K)

```bash
cp scripts/1.anneal.sh .
bash 1.anneal.sh
```

The particle is heated to 800 K — below the bulk melting point of Au (1337 K)
and Pd (1828 K), but well above the melting point of a 1–2 nm nanoparticle
due to Gibbs–Thomson depression. At this temperature atoms are mobile enough
to explore different chemical orderings without the particle evaporating.

After annealing, inspect the structure: has the chemical ordering changed
compared to the initial perfect cuboctahedron?


In [ ]:
atoms = read("structures/atoms1.xyz")
E0 = read("structures/atoms0.xyz").get_potential_energy()
E1 = atoms.get_potential_energy()
print(f"Energy after anneal : {E1:.3f} eV  ({E1/N_atoms:.4f} eV/atom)")
print(f"Change from initial : {E1-E0:+.3f} eV")

viewer = WeasWidget()
viewer.avr.model_style = 0
viewer.from_ase(atoms)
viewer


### Step 3 — Quench (800 K → 100 K)

```bash
cp scripts/2.quench.sh .
bash 2.quench.sh
```

The particle is rapidly cooled to 100 K. The disordered arrangement from
the anneal is kinetically frozen — this mimics a non-equilibrium disordered
alloy. Note that the energy typically *increases* slightly during quenching
because the system moves away from the 800 K basin without reaching the
100 K minimum.


In [ ]:
atoms = read("structures/atoms2.xyz")
E2 = atoms.get_potential_energy()
print(f"Energy after quench : {E2:.3f} eV  ({E2/N_atoms:.4f} eV/atom)")
print(f"Change from anneal  : {E2-E1:+.3f} eV")

viewer = WeasWidget()
viewer.avr.model_style = 0
viewer.from_ase(atoms)
viewer


### Step 4 — Relax (energy minimisation)

```bash
cp scripts/3.relax.sh .
bash 3.relax.sh
```

Geometry optimisation removes residual forces from the MD. The energy
should decrease noticeably compared to after the quench.

> **Question:** Compare your relaxed energy per atom with your neighbours.
> A lower energy per atom means a more stable chemical ordering was sampled
> during the anneal. What is the spread across the group?


In [ ]:
atoms = read("structures/atoms3.xyz")
E3 = atoms.get_potential_energy()
print(f"Energy after relax  : {E3:.3f} eV  ({E3/N_atoms:.4f} eV/atom)")
print(f"Change from quench  : {E3-E2:+.3f} eV")

viewer = WeasWidget()
viewer.avr.model_style = 0
viewer.from_ase(atoms)
viewer


### Step 5 — Second anneal (600 K)

```bash
cp scripts/4.anneal2.sh .
bash 4.anneal2.sh
```

A second, lower-temperature MD run allows further local relaxation.
This is the final "cooked" nanoparticle used for GCMC in Part 2.

After all five steps, plot the energy trajectory to confirm the particle
has stabilised.


In [ ]:
atoms = read("structures/atoms4.xyz")
E4 = atoms.get_potential_energy()
print(f"Energy after anneal2: {E4:.3f} eV  ({E4/N_atoms:.4f} eV/atom)")
print(f"Change from relax   : {E4-E3:+.3f} eV")
print(f"Total change        : {E4-E0:+.3f} eV  ({(E4-E0)/N_atoms:.4f} eV/atom)")

# Energy summary across cooking steps
labels = ['Initial', 'Anneal_800K', 'Quench_100K', 'Relax', 'Anneal_600K']
energies = [E0, E1, E2, E3, E4]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(5), [e/N_atoms for e in energies], 'o-',
        color='steelblue', lw=2, ms=8)
ax.set_xticks(range(5))
ax.set_xticklabels(labels, fontsize=10)
ax.set_ylabel('Energy per atom (eV)')
ax.set_title('Energy evolution during cooking')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

viewer = WeasWidget()
viewer.avr.model_style = 0
viewer.from_ase(atoms)
viewer


---
## Part 2 — Grand-canonical Monte Carlo (GCMC)

We now run GCMC to equilibrate hydrogen in the nanoparticle at fixed
chemical potential $\mu$, volume $V$, and temperature $T$ (the $\mu VT$
ensemble). At each MC step, TurboGAP randomly attempts:

- **Displacement** of a randomly chosen atom (probability $p_\mathrm{disp}$)
- **Insertion** of a H atom at a random position (probability $p_\mathrm{ins}$)
- **Removal** of a randomly chosen H atom (probability $p_\mathrm{rem}$)

with $p_\mathrm{disp} + p_\mathrm{ins} + p_\mathrm{rem} = 1$.

The chemical potential $\mu$ controls how strongly the reservoir "pushes"
hydrogen into the particle. More negative $\mu$ means lower H$_2$ pressure
and fewer H atoms at equilibrium.

### How to read the convergence plots

Two diagnostics confirm equilibration:

1. **Energy** — should plateau around a stable mean value.
2. **H concentration** $N_H / N_\mathrm{metal}$ — should fluctuate around
   a constant. If it is still rising or falling, run more MC steps.

The mean over the **last 50%** of the run is reported as the equilibrium value.

### GCMC-1 — your cooked nanoparticle


Copy and run the first GCMC script:
```bash
cp scripts/5.gcmc-1.sh .
bash 5.gcmc-1.sh
```
This runs GCMC on the annealed structure from Part 1 at a fixed $\mu$.
Visualise the trajectory to see H atoms inserted and removed.


In [ ]:
atoms_traj = read("5.gcmc-1/mc_all.xyz", index=':')
print(f"Saved MC frames : {len(atoms_traj)}")
print(f"H in last frame : {atoms_traj[-1].get_chemical_symbols().count('H')}")

viewer = WeasWidget()
viewer.avr.model_style = 0
viewer.from_ase(atoms_traj)
viewer


Plot energy and H concentration vs MC step to check convergence.


In [ ]:
N_metal = len(read("structures/atoms4.xyz"))  # metal atoms only

data   = np.genfromtxt('5.gcmc-1/mc.log')
nsteps = data[:, 0]
ener   = data[:, 4]
cH     = data[:, 7]

fig, axs = plt.subplots(1, 2, figsize=(9, 4), layout='constrained')

axs[0].plot(nsteps, ener, color='steelblue', lw=0.8)
axs[0].set_xlabel('MC steps')
axs[0].set_ylabel('Energy (eV)')
axs[0].set_title('Total energy — cooked NP')

axs[1].plot(nsteps, cH / N_metal * 100, color='darkorange', lw=0.8)
axs[1].set_xlabel('MC steps')
axs[1].set_ylabel('H / metal (%)')
axs[1].set_title('H concentration — cooked NP')

half = len(cH) // 2
mean_cH = cH[half:].mean() / N_metal * 100
axs[1].axhline(mean_cH, color='red', ls='--', lw=1,
               label=f'mean (last 50%) = {mean_cH:.1f}%')
axs[1].legend()

print(f"Mean H concentration (last 50%): {mean_cH:.1f} %")
plt.show()


### GCMC-2 — ordered cuboctahedron

We now compare against a reference structure: a **perfectly ordered** AuPd
cuboctahedron with 55 atoms (`Cu55_AuPd.xyz`).
In this structure Au and Pd occupy well-defined sublattice sites —
a chemically ordered intermetallic-like arrangement.

```bash
cp scripts/6.gcmc-2.sh .
bash 6.gcmc-2.sh
```

> **Before running:** predict whether the ordered or disordered particle
> will adsorb more H, based on the ensemble and ligand effects discussed
> in the Background section.


In [ ]:
atoms_traj = read("6.gcmc-2/mc_all.xyz", index=':')
print(f"Saved MC frames : {len(atoms_traj)}")
print(f"H in last frame : {atoms_traj[-1].get_chemical_symbols().count('H')}")

viewer = WeasWidget()
viewer.avr.model_style = 0
viewer.from_ase(atoms_traj)
viewer


In [ ]:
N_metal = 55

data   = np.genfromtxt('6.gcmc-2/mc.log')
nsteps = data[:, 0]
ener   = data[:, 4]
cH     = data[:, 7]

fig, axs = plt.subplots(1, 2, figsize=(9, 4), layout='constrained')

axs[0].plot(nsteps, ener, color='steelblue', lw=0.8)
axs[0].set_xlabel('MC steps')
axs[0].set_ylabel('Energy (eV)')
axs[0].set_title('Total energy — ordered CO')

axs[1].plot(nsteps, cH / N_metal * 100, color='darkorange', lw=0.8)
axs[1].set_xlabel('MC steps')
axs[1].set_ylabel('H / metal (%)')
axs[1].set_title('H concentration — ordered CO')

half = len(cH) // 2
mean_cH = cH[half:].mean() / N_metal * 100
axs[1].axhline(mean_cH, color='red', ls='--', lw=1,
               label=f'mean = {mean_cH:.1f}%')
axs[1].legend()
print(f"Mean H concentration (last 50%): {mean_cH:.1f} %")
plt.show()


### GCMC-3 — disordered cuboctahedron

Finally, compare against a **chemically disordered** cuboctahedron
(`Cu55_dis_AuPd.xyz`) with the same size and shape but randomised
Au/Pd site occupancy.

```bash
cp scripts/7.gcmc-3.sh .
bash 7.gcmc-3.sh
```

> **Question:** Does chemical ordering matter for hydrogen uptake?
> Compare the mean H concentrations from GCMC-2 and GCMC-3.
> Is your prediction from above confirmed? Which effect dominates?


In [ ]:
atoms_traj = read("7.gcmc-3/mc_all.xyz", index=':')
print(f"Saved MC frames : {len(atoms_traj)}")
print(f"H in last frame : {atoms_traj[-1].get_chemical_symbols().count('H')}")

viewer = WeasWidget()
viewer.avr.model_style = 0
viewer.from_ase(atoms_traj)
viewer


In [ ]:
N_metal = 55

fig, axs = plt.subplots(1, 2, figsize=(9, 4), layout='constrained')
results = {}

for folder, label, color in [
        ('6.gcmc-2', 'ordered',    'steelblue'),
        ('7.gcmc-3', 'disordered', 'darkorange')]:
    data   = np.genfromtxt(f'{folder}/mc.log')
    nsteps = data[:, 0]
    ener   = data[:, 4]
    cH     = data[:, 7]

    axs[0].plot(nsteps, ener, color=color, lw=0.8, label=label)
    axs[1].plot(nsteps, cH / N_metal * 100, color=color, lw=0.8, label=label)

    half = len(cH) // 2
    mean_cH = cH[half:].mean() / N_metal * 100
    results[label] = mean_cH
    print(f"{label:12s}  mean H conc. (last 50%): {mean_cH:.1f} %")

for ax, title, ylabel in zip(axs,
        ['Total energy', 'H concentration'],
        ['Energy (eV)', 'H / metal (%)']):
    ax.set_xlabel('MC steps')
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend()

plt.suptitle('Ordered vs disordered cuboctahedron', fontsize=12)
plt.tight_layout()
plt.show()

diff = results['disordered'] - results['ordered']
print(f"\nDifference (disordered − ordered): {diff:+.1f} %")


---
## Part 3 — H adsorption site analysis

Now we characterise *where* the H atoms sit on the nanoparticle.
Each H atom is classified by:

1. **Site type** — number of metal nearest neighbours
   (on-top=1, bridge=2, 3-fold hollow=3, 4-fold hollow=4, subsurface=≥5).
2. **Site composition** — which elements make up those neighbours
   (pure Pd, mixed PdAu, pure Au).
3. **Host coordination** — the coordination number (CN) of the metal atoms
   bonded to H, linking adsorption preference to surface site type
   (corner CN=5, edge CN=7, terrace CN=8–9).

The analysis is applied to the **final configuration** of any GCMC run.
Change `GCMC_FOLDER` below to switch between runs.


### Step 3.1 — Load the final configuration and build the neighbour list

We first load the structure and build a neighbour list with two cutoffs:

- **Metal–metal** (3.5 Å): used to compute coordination numbers.
- **H–metal** (2.5 Å): used to identify which metal atoms each H is bonded to.


In [ ]:
import re
from collections import Counter
from ase.io import read
from ase.neighborlist import NeighborList
import numpy as np

# ── change this to switch between GCMC runs ──────────────────────────────────
GCMC_FOLDER = '5.gcmc-1'
# ─────────────────────────────────────────────────────────────────────────────

atoms = read(f'{GCMC_FOLDER}/mc_current.xyz')
atoms.pbc = False

syms      = atoms.get_chemical_symbols()
n_atoms   = len(atoms)
h_idx     = [i for i, s in enumerate(syms) if s == 'H']
pd_idx    = [i for i, s in enumerate(syms) if s == 'Pd']
au_idx    = [i for i, s in enumerate(syms) if s == 'Au']
metal_idx = pd_idx + au_idx

print(f"Loaded : {GCMC_FOLDER}/mc_current.xyz")
print(f"Pd     : {len(pd_idx)}")
print(f"Au     : {len(au_idx)}")
print(f"H      : {len(h_idx)}")

CUTOFF_METAL = 3.5   # metal–metal coordination cutoff (Å)
CUTOFF_H     = 2.5   # H–metal bond cutoff (Å)

nl = NeighborList(
    [max(CUTOFF_METAL, CUTOFF_H) / 2] * n_atoms,
    self_interaction=False,
    bothways=True,
)
nl.update(atoms)
print("Neighbour list built.")


### Step 3.2 — Coordination numbers of metal atoms

The coordination number (CN) of each metal atom is the number of metal
neighbours within 3.5 Å. Atoms with CN < 11 are classified as **surface**
atoms; CN ≥ 11 are **bulk-like**.

This classification will be used in Step 3.3 to identify subsurface H.


In [ ]:
coord_numbers = np.zeros(n_atoms, dtype=int)
for i in metal_idx:
    nbrs, _ = nl.get_neighbors(i)
    metal_nbrs = [
        n for n in nbrs
        if syms[n] in ('Pd', 'Au')
        and np.linalg.norm(atoms.positions[i] - atoms.positions[n]) < CUTOFF_METAL
    ]
    coord_numbers[i] = len(metal_nbrs)

surf_idx = [i for i in metal_idx if coord_numbers[i] < 11]
bulk_idx = [i for i in metal_idx if coord_numbers[i] >= 11]

print(f"Surface metal atoms : {len(surf_idx)}  "
      f"(mean CN = {np.mean([coord_numbers[i] for i in surf_idx]):.2f})")
print(f"Bulk metal atoms    : {len(bulk_idx)}  "
      f"(mean CN = {np.mean([coord_numbers[i] for i in bulk_idx]):.2f})")

# Print full CN distribution
cn_counter = Counter(coord_numbers[i] for i in metal_idx)
print("\nCN distribution:")
print(f"  {'CN':>4}  {'count':>6}  {'site type'}")
site_names = {5: 'corner', 6: 'corner/edge', 7: 'edge',
              8: '(100) terrace', 9: '(111) terrace',
              10: 'near-surface', 12: 'bulk'}
for cn, count in sorted(cn_counter.items()):
    print(f"  {cn:>4}  {count:>6}  {site_names.get(cn, '')}")


In [ ]:
# Visualise CN distribution for Pd vs Au separately
fig, ax = plt.subplots(figsize=(7, 4))

for elem, idx_list, color, marker in [
        ('Pd', pd_idx, 'steelblue', 'o'),
        ('Au', au_idx, 'goldenrod', 's')]:
    cns = [coord_numbers[i] for i in idx_list]
    if cns:
        cn_vals = sorted(set(cns))
        counts  = [cns.count(c) for c in cn_vals]
        ax.plot(cn_vals, counts, f'{marker}-', color=color,
                lw=2, ms=7, label=elem)

ax.axvline(11, color='red', ls='--', lw=1.2, label='surface/bulk (CN=11)')
ax.set_xlabel('Coordination number')
ax.set_ylabel('Number of atoms')
ax.set_title('CN distribution — Pd vs Au')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### Step 3.3 — Classify H atoms by site type and composition

For each H atom we find its metal neighbours within 2.5 Å and record:
- how many there are (→ site type)
- which elements they are (→ site composition)
- whether they are all bulk-like (→ subsurface H)


In [ ]:
def normalize_comp(symbols):
    """Sort element symbols alphabetically, e.g. ['Pd','Au'] → 'AuPd'."""
    return ''.join(sorted(symbols))

site_data = []

for hi in h_idx:
    nbrs, _ = nl.get_neighbors(hi)
    metal_nbrs = [
        n for n in nbrs
        if syms[n] in ('Pd', 'Au')
        and np.linalg.norm(atoms.positions[hi] - atoms.positions[n]) < CUTOFF_H
    ]
    n_metal = len(metal_nbrs)
    comp_symbols = [syms[n] for n in metal_nbrs]
    comp_str = normalize_comp(comp_symbols)

    # classify site type by number of metal neighbours
    site_map = {0: 'unbound', 1: 'ontop', 2: 'bridge',
                3: 'hollow3', 4: 'hollow4'}
    site = site_map.get(n_metal, 'subsurface')

    # override: if all metal neighbours are bulk-like → subsurface
    if n_metal > 0 and all(coord_numbers[n] >= 11 for n in metal_nbrs):
        site = 'subsurface'

    bond_lengths = [
        np.linalg.norm(atoms.positions[hi] - atoms.positions[n])
        for n in metal_nbrs
    ]

    site_data.append({
        'h_index'   : hi,
        'site'      : site,
        'n_metal'   : n_metal,
        'n_Pd'      : comp_symbols.count('Pd'),
        'n_Au'      : comp_symbols.count('Au'),
        'comp'      : comp_str if comp_str else 'none',
        'mean_bond' : np.mean(bond_lengths) if bond_lengths else 0.0,
    })

# Summary
by_site = Counter(d['site'] for d in site_data)
by_comp = Counter(d['comp'] for d in site_data)

print(f"Total H atoms classified: {len(site_data)}")
print("\n--- Site type ---")
for k, v in sorted(by_site.items()):
    pct = v / len(site_data) * 100
    print(f"  {k:15s}: {v:3d}  ({pct:.0f}%)")

print("\n--- Site composition ---")
for k, v in sorted(by_comp.items()):
    pct = v / len(site_data) * 100
    print(f"  {k:20s}: {v:3d}  ({pct:.0f}%)")

print(f"\nMean Pd neighbours per H : "
      f"{np.mean([d['n_Pd'] for d in site_data]):.2f}")
print(f"Mean Au neighbours per H : "
      f"{np.mean([d['n_Au'] for d in site_data]):.2f}")
print(f"Mean H–metal bond length : "
      f"{np.mean([d['mean_bond'] for d in site_data if d['mean_bond']>0]):.3f} Å")


### Step 3.4 — Bar charts: site type and composition

Which site types are most populated? Is H predominantly on Pd, Au,
or mixed sites?


In [ ]:
import matplotlib.patches as mpatches

SITE_COLORS = {
    'ontop'     : 'orchid',
    'bridge'    : 'coral',
    'hollow3'   : 'steelblue',
    'hollow4'   : 'goldenrod',
    'subsurface': 'dimgray',
    'unbound'   : 'black',
}

def comp_color(c):
    elems = re.findall(r'[A-Z][a-z]?', c)
    if all(e == 'Pd' for e in elems): return 'steelblue'
    if all(e == 'Au' for e in elems): return 'goldenrod'
    return 'mediumseagreen'

fig, axes = plt.subplots(1, 2, figsize=(12, 4), layout='constrained')

# Site type bar
ax = axes[0]
sites  = sorted(by_site.keys())
counts = [by_site[s] for s in sites]
colors = [SITE_COLORS.get(s, 'gray') for s in sites]
bars   = ax.bar(sites, counts, color=colors, alpha=0.85, edgecolor='white', width=0.6)
ax.bar_label(bars, fontsize=10)
ax.set_ylabel('Number of H atoms')
ax.set_title('H site types')
ax.set_xticklabels(sites, rotation=25, ha='right')
patches = [mpatches.Patch(color=c, label=s)
           for s, c in SITE_COLORS.items() if s in by_site]
ax.legend(handles=patches, fontsize=8, loc='upper right')

# Composition bar
ax = axes[1]
comps  = sorted(by_comp.keys())
counts = [by_comp[c] for c in comps]
colors = [comp_color(c) for c in comps]
bars   = ax.bar(comps, counts, color=colors, alpha=0.85, edgecolor='white', width=0.6)
ax.bar_label(bars, fontsize=10)
ax.set_ylabel('Number of H atoms')
ax.set_title('H site compositions')
ax.set_xticklabels(comps, rotation=25, ha='right')
patches = [
    mpatches.Patch(color='steelblue',      label='pure Pd'),
    mpatches.Patch(color='mediumseagreen', label='mixed PdAu'),
    mpatches.Patch(color='goldenrod',      label='pure Au'),
]
ax.legend(handles=patches, fontsize=8)

plt.suptitle(f'H adsorption sites — {GCMC_FOLDER}', fontsize=12)
plt.show()


### Step 3.5 — Heatmap: site type × composition

The heatmap shows which combinations of site type and composition
are most frequently occupied. For example, do H atoms prefer
3-fold hollow sites on pure Pd, or on mixed PdAu sites?


In [ ]:
sites_list = sorted(set(d['site'] for d in site_data))
comps_list = sorted(set(d['comp'] for d in site_data))

mat = np.zeros((len(sites_list), len(comps_list)), dtype=int)
for d in site_data:
    if d['site'] in sites_list and d['comp'] in comps_list:
        mat[sites_list.index(d['site'])][comps_list.index(d['comp'])] += 1

fig, ax = plt.subplots(figsize=(max(6, len(comps_list)*1.2),
                                max(4, len(sites_list)*0.9)))
im = ax.imshow(mat, cmap='Blues', aspect='auto')
ax.set_xticks(range(len(comps_list)))
ax.set_yticks(range(len(sites_list)))
ax.set_xticklabels(comps_list, rotation=30, ha='right', fontsize=10)
ax.set_yticklabels(sites_list, fontsize=10)
ax.set_xlabel('Site composition')
ax.set_ylabel('Site type')
ax.set_title(f'Site type × composition — {GCMC_FOLDER}')

for i in range(len(sites_list)):
    for j in range(len(comps_list)):
        if mat[i, j] > 0:
            ax.text(j, i, str(mat[i, j]),
                    ha='center', va='center', fontsize=11,
                    color='white' if mat[i, j] > mat.max() * 0.5 else 'black')

plt.colorbar(im, ax=ax, label='count')
plt.tight_layout()
plt.show()


### Step 3.6 — Which metal sites preferentially bind H?

Here we ask: for the metal atoms that *are* bonded to H, what is their
coordination number? If H prefers low-CN sites (edges, corners),
the distribution should be shifted towards smaller CN values compared
to the overall metal CN distribution.

This is the quantitative link between adsorption preference and
nanoparticle geometry.


In [ ]:
# Collect CN of all metal atoms bonded to at least one H
h_bonded_cn   = []
h_bonded_elem = []

for d in site_data:
    hi   = d['h_index']
    nbrs, _ = nl.get_neighbors(hi)
    metal_nbrs = [
        n for n in nbrs
        if syms[n] in ('Pd', 'Au')
        and np.linalg.norm(atoms.positions[hi] - atoms.positions[n]) < CUTOFF_H
    ]
    for n in metal_nbrs:
        h_bonded_cn.append(coord_numbers[n])
        h_bonded_elem.append(syms[n])

fig, axes = plt.subplots(1, 2, figsize=(12, 4), layout='constrained')

# Left: all metal CN vs H-bonded metal CN
ax = axes[0]
cn_all = [coord_numbers[i] for i in metal_idx]
cn_h   = h_bonded_cn

for data_cn, label, color, ls in [
        (cn_all, 'all metal', 'gray', '--'),
        (cn_h,   'bonded to H', 'steelblue', '-')]:
    cn_vals = sorted(set(data_cn))
    counts  = [data_cn.count(c) / len(data_cn) for c in cn_vals]  # normalised
    ax.plot(cn_vals, counts, ls, color=color, lw=2, ms=6,
            marker='o', label=label)

ax.axvline(11, color='red', ls=':', lw=1.2, label='surface/bulk (CN=11)')
ax.set_xlabel('Coordination number')
ax.set_ylabel('Fraction of atoms')
ax.set_title('CN: all metal vs H-bonded metal')
ax.legend()
ax.grid(True, alpha=0.3)

# Right: Pd vs Au among H-bonded atoms
ax = axes[1]
for elem, color, marker in [('Pd', 'steelblue', 'o'), ('Au', 'goldenrod', 's')]:
    cns = [cn for cn, el in zip(h_bonded_cn, h_bonded_elem) if el == elem]
    if cns:
        cn_vals = sorted(set(cns))
        counts  = [cns.count(c) for c in cn_vals]
        ax.plot(cn_vals, counts, f'{marker}-', color=color,
                lw=2, ms=6, label=elem)

ax.axvline(11, color='red', ls=':', lw=1.2, label='surface/bulk (CN=11)')
ax.set_xlabel('Coordination number of metal atom bonded to H')
ax.set_ylabel('Count')
ax.set_title('CN of H-bonded atoms: Pd vs Au')
ax.legend()
ax.grid(True, alpha=0.3)

plt.suptitle(f'Coordination analysis — {GCMC_FOLDER}', fontsize=12)
plt.tight_layout()
plt.show()

# Print mean CN per element
for elem in ('Pd', 'Au'):
    cns = [cn for cn, el in zip(h_bonded_cn, h_bonded_elem) if el == elem]
    if cns:
        print(f"Mean CN of {elem} atoms bonded to H: {np.mean(cns):.2f}")


### Step 3.7 — Export coloured structure for visualisation

Write an XYZ file where each H atom carries a `site_type` property
(integer label). This can be loaded in OVITO or VESTA and coloured
by site type to visualise the spatial distribution of adsorption sites.

| `site_type` | Site |
|-------------|------|
| 0 | metal (not H) |
| 1 | on-top |
| 2 | bridge |
| 3 | 3-fold hollow |
| 4 | 4-fold hollow |
| 5 | subsurface |
| 6 | unbound |


In [ ]:
SITE_LABEL = {
    'ontop': 1, 'bridge': 2, 'hollow3': 3,
    'hollow4': 4, 'subsurface': 5, 'unbound': 6,
}

h_site_type = {d['h_index']: SITE_LABEL.get(d['site'], 0)
               for d in site_data}

outfile = f'{GCMC_FOLDER}_H_colored.xyz'
with open(f'{GCMC_FOLDER}/mc_current.xyz') as f:
    lines = f.readlines()

n_file = int(lines[0].strip())
with open(outfile, 'w') as f:
    f.write(f"{n_file}\n")
    f.write(lines[1].strip() +
            " Properties=species:S:1:pos:R:3:site_type:I:1\n")
    for i, line in enumerate(lines[2:2 + n_file]):
        f.write(f"{line.rstrip()} {h_site_type.get(i, 0)}\n")

print(f"Saved: {outfile}")
print("Load in OVITO and colour by 'site_type' to visualise.")

# Quick weas_widget preview coloured by element
from weas_widget import WeasWidget
atoms_col = read(outfile)
viewer = WeasWidget()
viewer.avr.model_style = 0
viewer.from_ase(atoms_col)
viewer


### Step 3.8 — Compare site populations across GCMC runs

Run the analysis on GCMC-1, GCMC-2, and GCMC-3 and compare the
site-type distributions side by side. Does the ordered particle
show a different site preference than the disordered one?

> **Expected trend:** if the ordered particle has more isolated Pd sites
> (surrounded by Au), you should see fewer pure-Pd hollow sites and
> more mixed PdAu sites compared to the disordered particle.


In [ ]:
def analyse_sites(filepath, cutoff_metal=3.5, cutoff_H=2.5):
    """Return (by_site, by_comp) Counter dicts for a given xyz file."""
    atoms = read(filepath)
    atoms.pbc = False
    syms    = atoms.get_chemical_symbols()
    n       = len(atoms)
    h_idx   = [i for i, s in enumerate(syms) if s == 'H']
    m_idx   = [i for i, s in enumerate(syms) if s in ('Pd', 'Au')]

    nl = NeighborList([max(cutoff_metal, cutoff_H) / 2] * n,
                      self_interaction=False, bothways=True)
    nl.update(atoms)

    cn = np.zeros(n, dtype=int)
    for i in m_idx:
        nbrs, _ = nl.get_neighbors(i)
        cn[i] = sum(1 for nb in nbrs
                    if syms[nb] in ('Pd', 'Au')
                    and np.linalg.norm(atoms.positions[i] -
                                       atoms.positions[nb]) < cutoff_metal)

    by_site, by_comp = Counter(), Counter()
    for hi in h_idx:
        nbrs, _ = nl.get_neighbors(hi)
        mnbrs = [nb for nb in nbrs
                 if syms[nb] in ('Pd', 'Au')
                 and np.linalg.norm(atoms.positions[hi] -
                                    atoms.positions[nb]) < cutoff_H]
        n_m  = len(mnbrs)
        comp = normalize_comp([syms[nb] for nb in mnbrs])
        site_map = {0:'unbound',1:'ontop',2:'bridge',3:'hollow3',4:'hollow4'}
        site = site_map.get(n_m, 'subsurface')
        if n_m > 0 and all(cn[nb] >= 11 for nb in mnbrs):
            site = 'subsurface'
        by_site[site] += 1
        by_comp[comp if comp else 'none'] += 1
    return by_site, by_comp

# Run for each GCMC folder
run_info = [
    ('5.gcmc-1', 'cooked'),
    ('6.gcmc-2', 'ordered'),
    ('7.gcmc-3', 'disordered'),
]

all_results = {}
for folder, label in run_info:
    try:
        bs, bc = analyse_sites(f'{folder}/mc_current.xyz')
        all_results[label] = (bs, bc)
        print(f"{label:12s}: {sum(bs.values())} H atoms")
    except FileNotFoundError:
        print(f"{label:12s}: file not found — run the GCMC first")

# Plot side-by-side site type comparison
all_sites = sorted(set(s for bs, _ in all_results.values() for s in bs))
x = np.arange(len(all_sites))
width = 0.25
colors_run = ['steelblue', 'darkorange', 'seagreen']

fig, ax = plt.subplots(figsize=(10, 4))
for i, (label, (bs, _)) in enumerate(all_results.items()):
    total = sum(bs.values()) or 1
    counts = [bs.get(s, 0) / total * 100 for s in all_sites]
    bars = ax.bar(x + i * width, counts, width,
                  label=label, color=colors_run[i], alpha=0.85)

ax.set_xticks(x + width)
ax.set_xticklabels(all_sites, rotation=25, ha='right')
ax.set_ylabel('H atoms (%)')
ax.set_title('H site type distribution — ordered vs disordered')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()


---
## Part 4 — Parameter sensitivity

Explore how GCMC parameters affect the equilibrium H concentration and
convergence speed. For each experiment, modify the relevant parameter
in the run script, re-run, and add the result to the comparison plot below.

| Parameter | TurboGAP keyword | Suggested values |
|-----------|-----------------|-----------------|
| Max displacement (Å) | `mc_move_max` | 0.05, 0.20, 0.80 |
| Temperature (K) | `temperature` | 200, 300, 500 |
| Chemical potential (eV) | `mu` | vary by ±0.2 eV |
| Move ratio | `mc_moves` | insertion-only vs mixed |

> **Question 1:** Which parameter affects the *equilibrium* H concentration?
> Which affects only the *convergence speed*?

> **Question 2:** Sketch qualitatively what an adsorption isotherm
> $\langle N_H\rangle(\mu)$ would look like for this system.
> At what value of $\mu$ do you expect it to saturate?


In [ ]:
# Template — add your run directories and labels here
N_metal = 55
folders = ['5.gcmc-1']   # extend as you add runs
labels  = ['baseline']
colors  = ['steelblue', 'darkorange', 'seagreen', 'orchid', 'coral']

fig, axs = plt.subplots(1, 2, figsize=(9, 4), layout='constrained')

for folder, label, color in zip(folders, labels, colors):
    data   = np.genfromtxt(f'{folder}/mc.log')
    nsteps = data[:, 0]
    ener   = data[:, 4]
    cH     = data[:, 7]

    axs[0].plot(nsteps, ener, color=color, lw=0.8, label=label)
    axs[1].plot(nsteps, cH / N_metal * 100, color=color, lw=0.8, label=label)

    half = len(cH) // 2
    print(f"{label:20s}  mean H conc.: "
          f"{cH[half:].mean()/N_metal*100:.1f} %")

axs[0].set_xlabel('MC steps'); axs[0].set_ylabel('Energy (eV)')
axs[0].set_title('Energy convergence'); axs[0].legend()
axs[1].set_xlabel('MC steps'); axs[1].set_ylabel('H / metal (%)')
axs[1].set_title('H concentration'); axs[1].legend()
plt.tight_layout()
plt.show()
